# Advanced Python Object Representations: `__repr__`, `__str__`, and Beyond

**One executable, problem-driven Jupyter notebook | 28 worked problems including the capstone | Python 3.10+ | standard library only**

This notebook expands the supplied `Person` lesson into increasingly demanding tasks: representation dispatch; the absence of one or both methods; formatting conversions; correct escaping; containers; inheritance; dataclasses; exceptions; format specifications; privacy-sensitive diagnostic output; recursive graphs; stable summaries; logging; reconstructible expressions; serialization; and a final integrated project.

**Study method:** Read a problem and predict the behavior, implement it yourself if desired, then compare with the worked solution and run the assertions. Run **Kernel → Restart & Run All** for a clean verification. Each problem uses its own class names to avoid unintended redefinitions. All execution is local; no network, installation, external files, or unsafe `eval()` are required.

**Accuracy note on the supplied lesson:** It includes a `Person.__repr__` f-string with misplaced quotation marks and, in a later example, a literal `self.age` instead of the attribute value. This notebook intentionally corrects those examples using `!r`, e.g. `f"Person(name={self.name!r}, age={self.age!r})"`. The original lesson's account of the `repr`/`str` fallback is retained and extended, with separate attention to `object.__format__` and notebook display behavior.

**Terminology:** `repr(x)` is principally a developer-facing representation; `str(x)` is principally a human-facing representation. A constructor-looking `repr` is useful *when practical*, not mandatory, and must never be assumed safe to evaluate.

## Quick reference — predict which method runs

| Operation | Primary behavior |
|---|---|
| `repr(x)`, `f"{x!r}"`, `%r` | `type(x).__repr__(x)` |
| `str(x)`, `print(x)`, `f"{x!s}"`, `%s` | `type(x).__str__(x)`; inherited `object.__str__` delegates to `__repr__` |
| `f"{x}"`, `format(x, "")` | `type(x).__format__(x, "")`; the inherited implementation uses `str(x)` for an empty spec |
| `f"{x:spec}"` | `__format__(x, "spec")`; the inherited `object.__format__` rejects a nonempty spec |
| `ascii(x)`, `f"{x!a}"` | `repr(x)` with non-ASCII characters escaped |
| `[x]`, `{"k": x}` | Containers ordinarily use contained objects' `repr` |
| Jupyter's default *text/plain* rendering of a bare result | Commonly `repr`-based, but rich display hooks and formatter configuration can override it |

**Important:** Special methods are generally resolved on the object's **type**, not by an instance attribute with the same name. `repr` is **not** a secure serialization format. Default object representations often contain an address-like value that changes between runs.

In [1]:
import ast
import io
import itertools
import json
import logging
import re
import reprlib
from dataclasses import dataclass, field
from decimal import Decimal, InvalidOperation
from typing import Any

print("Ready: standard-library imports successful.")

Ready: standard-library imports successful.


## Problem 01 — Dispatch matrix and fallbacks

**Task.** Create four classes: no custom representation, only `__repr__`, only `__str__`, and both. Verify the behavior of `repr`, `str`, `print` (via `str`), and empty formatting. Avoid asserting a specific memory address.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 01 — reasoning

When only `__repr__` is implemented, the inherited `object.__str__` forwards to it. Conversely, `repr` never falls back to a custom `__str__`; it inherits the default `object.__repr__`. Use structural checks for address-bearing defaults rather than snapshotting the address.

In [2]:
class Plain01:
    pass

class ReprOnly01:
    def __repr__(self) -> str:
        return "<repr-only>"

class StrOnly01:
    def __str__(self) -> str:
        return "str-only"

class Both01:
    def __repr__(self) -> str:
        return "<both:developer>"

    def __str__(self) -> str:
        return "both:user"

objects01 = [Plain01(), ReprOnly01(), StrOnly01(), Both01()]
for obj in objects01:
    print(f"{type(obj).__name__:>12}: repr={repr(obj)!r}; str={str(obj)!r}; empty format={format(obj, '')!r}")

     Plain01: repr='<__main__.Plain01 object at 0x000001BB4A634440>'; str='<__main__.Plain01 object at 0x000001BB4A634440>'; empty format='<__main__.Plain01 object at 0x000001BB4A634440>'
  ReprOnly01: repr='<repr-only>'; str='<repr-only>'; empty format='<repr-only>'
   StrOnly01: repr='<__main__.StrOnly01 object at 0x000001BB5A61E900>'; str='str-only'; empty format='str-only'
      Both01: repr='<both:developer>'; str='both:user'; empty format='both:user'


### Verification 01 — executable tests

In [3]:
plain, repr_only, str_only, both = objects01
assert repr(repr_only) == str(repr_only) == "<repr-only>"
assert repr(both) == "<both:developer>" and str(both) == "both:user"
assert str(str_only) == "str-only"
assert type(str_only).__name__ in repr(str_only)
assert "str-only" not in repr(str_only)
assert str(plain) == repr(plain)
assert format(both, "") == str(both)
print("PASS 01 — representation dispatch and fallback")

PASS 01 — representation dispatch and fallback


**Extension / discussion.** Why is a test such as `repr(Plain01()) == "<__main__.Plain01 object at 0x1234>"` inherently fragile?

## Problem 02 — Contract violations and side effects

**Task.** Demonstrate that both representation methods must return `str`. Capture the resulting `TypeError`s. Then write a side-effect-free, correctly typed alternative.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 02 — reasoning

The interpreter does not automatically coerce an integer or `None` returned from a representation method. Methods that print debug messages also modify output as an incidental side effect; keep production representations pure where possible.

In [4]:
class BrokenRepr02:
    def __repr__(self):
        return 42

class BrokenStr02:
    def __str__(self):
        return None

class SafeRecord02:
    def __init__(self, value: int) -> None:
        self.value = value

    def __repr__(self) -> str:
        return f"SafeRecord02(value={self.value!r})"

    def __str__(self) -> str:
        return f"record {self.value}"

def captured_type_error(function) -> str:
    try:
        function()
    except TypeError as exc:
        return str(exc)
    raise AssertionError("TypeError was expected")

print("Invalid repr:", captured_type_error(lambda: repr(BrokenRepr02())))
print("Invalid str:", captured_type_error(lambda: str(BrokenStr02())))

Invalid repr: __repr__ returned non-string (type int)
Invalid str: __str__ returned non-string (type NoneType)


### Verification 02 — executable tests

In [5]:
assert "non-string" in captured_type_error(lambda: repr(BrokenRepr02()))
assert "non-string" in captured_type_error(lambda: str(BrokenStr02()))
record02 = SafeRecord02(7)
assert repr(record02) == "SafeRecord02(value=7)"
assert str(record02) == "record 7"
assert repr(record02) == repr(record02)  # stable if state stays unchanged
print("PASS 02 — return types and pure output")

PASS 02 — return types and pure output


## Problem 03 — Correct quoting for hostile-looking text

**Task.** Correct the source lesson's malformed `Person` representation. Test names containing single quotes, double quotes, backslashes, tabs, and newlines, plus an age. Do not build an ad hoc escaping function.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 03 — reasoning

The `!r` conversion embeds valid Python-literal-style quoting for ordinary string fields and avoids hard-coding a particular quote style. It handles special characters much more reliably than `name='{self.name}'`. `str` can remain compact and readable.

In [6]:
class Person03:
    def __init__(self, name: str, age: int) -> None:
        if not isinstance(name, str):
            raise TypeError("name must be str")
        if type(age) is not int or age < 0:
            raise ValueError("age must be a nonnegative integer")
        self.name, self.age = name, age

    def __repr__(self) -> str:
        return f"Person03(name={self.name!r}, age={self.age!r})"

    def __str__(self) -> str:
        return f"{self.name} ({self.age})"

names03 = ["Python", "O'Reilly", 'a "quote"', "path\to\file", "two\nlines", "tab\tstop"]
people03 = [Person03(name, 30) for name in names03]
for item in people03:
    print("debug:", repr(item), "| display:", str(item))

debug: Person03(name='Python', age=30) | display: Python (30)
debug: Person03(name="O'Reilly", age=30) | display: O'Reilly (30)
debug: Person03(name='a "quote"', age=30) | display: a "quote" (30)
debug: Person03(name='path\to\x0cile', age=30) | display: path	oile (30)
debug: Person03(name='two\nlines', age=30) | display: two
lines (30)
debug: Person03(name='tab\tstop', age=30) | display: tab	stop (30)


### Verification 03 — executable tests

In [7]:
for person in people03:
    representation = repr(person)
    assert representation.startswith("Person03(name=")
    assert repr(person.name) in representation
    assert "age=30" in representation
    assert person.name in str(person)
assert "\\n" in repr(Person03("line\nbreak", 1))
try:
    Person03("Test", -1)
except ValueError:
    pass
else:
    raise AssertionError("Negative age should be rejected")
print("PASS 03 — quoting, control characters, validation")

PASS 03 — quoting, control characters, validation


**Extension / discussion.** To preserve an argument as a literal in generated Python-like text, prefer `!r` over manually inserting quotes around it.

## Problem 04 — Conversion flags: `!s`, `!r`, `!a`

**Task.** Predict `f"{value}"`, `f"{value!s}"`, `f"{value!r}"`, `f"{value!a}"`, `%s`, and `%r` for a value containing non-ASCII text.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 04 — reasoning

`!s` explicitly requests `str`, `!r` requests `repr`, and `!a` uses `ascii(repr-like-output)` with non-ASCII escapes. `%s` and `%r` are the corresponding legacy formatting conversions. An ordinary f-string expression with no explicit conversion goes through `__format__`.

In [8]:
class Label04:
    def __init__(self, text: str) -> None:
        self.text = text

    def __repr__(self) -> str:
        return f"Label04(text={self.text!r})"

    def __str__(self) -> str:
        return self.text

label04 = Label04("café ☕")
formats04 = {
    "default": f"{label04}",
    "!s": f"{label04!s}",
    "!r": f"{label04!r}",
    "!a": f"{label04!a}",
    "%s": "%s" % label04,
    "%r": "%r" % label04,
}
for method, result in formats04.items():
    print(f"{method:>8} -> {result}")

 default -> café ☕
      !s -> café ☕
      !r -> Label04(text='café ☕')
      !a -> Label04(text='caf\xe9 \u2615')
      %s -> café ☕
      %r -> Label04(text='café ☕')


### Verification 04 — executable tests

In [9]:
assert formats04["default"] == formats04["!s"] == formats04["%s"] == "café ☕"
assert formats04["!r"] == formats04["%r"] == repr(label04)
assert formats04["!a"] == ascii(label04)
assert "\\xe9" in formats04["!a"] and "\\u2615" in formats04["!a"]
assert "café" not in formats04["!a"]
print("PASS 04 — all conversion flags")

PASS 04 — all conversion flags


## Problem 05 — Nested containers and element representations

**Task.** Create a nested dictionary/list containing objects with distinct `__str__` and `__repr__`. Explain the difference between printing an element and printing its containing list or dictionary.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 05 — reasoning

Built-in containers display their elements with `repr`, since nested diagnostic values should remain distinguishable. A string nested inside a container also displays with quotes. A single element printed directly uses `str`.

In [10]:
class Package05:
    def __init__(self, sku: str, units: int) -> None:
        self.sku, self.units = sku, units

    def __repr__(self) -> str:
        return f"Package05(sku={self.sku!r}, units={self.units!r})"

    def __str__(self) -> str:
        return f"{self.sku} × {self.units}"

package05 = Package05("X-12", 4)
inventory05 = {"shipment": [package05, "fragile"]}
print("single:", package05)
print("list:", inventory05["shipment"])
print("dict:", inventory05)

single: X-12 × 4
list: [Package05(sku='X-12', units=4), 'fragile']
dict: {'shipment': [Package05(sku='X-12', units=4), 'fragile']}


### Verification 05 — executable tests

In [11]:
assert str(package05) == "X-12 × 4"
assert repr(package05) in repr(inventory05)
assert "'fragile'" in repr(inventory05)
assert "X-12 × 4" not in repr(inventory05)
assert f"{package05!r}" == repr(package05)
print("PASS 05 — nested representations")

PASS 05 — nested representations


## Problem 06 — Special-method lookup lives on the type

**Task.** Set an instance attribute named `__repr__` and prove it does not change `repr(instance)`. Then call that attribute explicitly. Avoid modifying a shared class's special methods.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 06 — reasoning

Implicit special-method invocation performs type-level lookup. Overriding `instance.__repr__` in its dictionary can shadow explicit attribute access, but not the interpreter's `repr(instance)` dispatch. This distinction extends to many other dunder operations.

In [12]:
class Token06:
    def __repr__(self) -> str:
        return "Token06(original=True)"

token06 = Token06()
before06 = repr(token06)
token06.__repr__ = lambda: "INSTANCE-ONLY"  # instance dictionary entry
explicit06 = token06.__repr__()
implicit06 = repr(token06)
print("explicit:", explicit06)
print("implicit:", implicit06)

explicit: INSTANCE-ONLY
implicit: Token06(original=True)


### Verification 06 — executable tests

In [13]:
assert before06 == implicit06 == "Token06(original=True)"
assert explicit06 == "INSTANCE-ONLY"
assert "__repr__" in vars(token06)
assert type(token06).__repr__(token06) == implicit06
print("PASS 06 — implicit type-level dispatch")

PASS 06 — implicit type-level dispatch


## Problem 07 — Subclassing without losing information

**Task.** Design `Event` and `TimedEvent` with useful diagnostic and readable views. Preserve the subclass name and its additional `timestamp` attribute. Explain when a base class using `type(self).__name__` is insufficient.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 07 — reasoning

A generic class-name substitution is not enough to make a subclass repr reconstructible: the subclass may need additional constructor arguments. Give a subclass its own `__repr__`; inherit or override `__str__` based on what users should see.

In [14]:
class Event07:
    def __init__(self, title: str) -> None:
        self.title = title

    def __repr__(self) -> str:
        return f"Event07(title={self.title!r})"

    def __str__(self) -> str:
        return self.title

class TimedEvent07(Event07):
    def __init__(self, title: str, timestamp: str) -> None:
        super().__init__(title)
        self.timestamp = timestamp

    def __repr__(self) -> str:
        return f"TimedEvent07(title={self.title!r}, timestamp={self.timestamp!r})"

    def __str__(self) -> str:
        return f"{self.title} @ {self.timestamp}"

event07 = TimedEvent07("deploy", "2026-09-19T10:15:00Z")
print(repr(event07))
print(str(event07))

TimedEvent07(title='deploy', timestamp='2026-09-19T10:15:00Z')
deploy @ 2026-09-19T10:15:00Z


### Verification 07 — executable tests

In [15]:
assert isinstance(event07, Event07)
assert repr(event07) == "TimedEvent07(title='deploy', timestamp='2026-09-19T10:15:00Z')"
assert "2026-09-19" in str(event07)
assert "timestamp=" not in repr(Event07("basic"))
assert type(event07).__name__ in repr(event07)
print("PASS 07 — subclass-aware representations")

PASS 07 — subclass-aware representations


## Problem 08 — Dataclass-generated representation

**Task.** Compare a generated dataclass `__repr__` with a custom human-readable `__str__`. Exclude an internal cache from the generated repr and verify that `repr=False` is only a presentation choice, not access control.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 08 — reasoning

`@dataclass` generates a field-oriented `__repr__` unless told otherwise. `field(repr=False)` excludes a field from that generated repr; it does not erase or protect the value, and explicit custom code may still expose it.

In [16]:
@dataclass
class Report08:
    title: str
    pages: int
    _cache: dict[str, str] = field(default_factory=dict, repr=False, compare=False)

    def __str__(self) -> str:
        return f"{self.title}: {self.pages} pages"

report08 = Report08("Annual", 12)
report08._cache["raw"] = "internal"
print("generated repr:", repr(report08))
print("custom str:", str(report08))

generated repr: Report08(title='Annual', pages=12)
custom str: Annual: 12 pages


### Verification 08 — executable tests

In [17]:
assert repr(report08) == "Report08(title='Annual', pages=12)"
assert str(report08) == "Annual: 12 pages"
assert "_cache" not in repr(report08)
assert report08._cache["raw"] == "internal"
assert report08 == Report08("Annual", 12, {"different": "cache"})
print("PASS 08 — dataclass repr configuration")

PASS 08 — dataclass repr configuration


## Problem 09 — Protect against accidental secret disclosure

**Task.** Build an `ApiCredential` object. Mask its key in both `__repr__` and `__str__`; show safe output in a list and a dictionary. State explicitly what representation redaction does *not* guarantee.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 09 — reasoning

Never print the full token while demonstrating a masked representation. A descriptive placeholder is safer than a clever substring because even token fragments can be sensitive. Redacting representations reduces accidental disclosure; it is not encryption, zeroization, or protection against inspecting accessible attributes.

In [18]:
class ApiCredential09:
    def __init__(self, service: str, token: str) -> None:
        self.service = service
        self._token = token

    def __repr__(self) -> str:
        return f"ApiCredential09(service={self.service!r}, token=<redacted>)"

    def __str__(self) -> str:
        return f"Credential for {self.service}"

credential09 = ApiCredential09("billing", "DEMO_TOKEN_DO_NOT_LOG_42")
views09 = [str(credential09), repr(credential09), repr([credential09]), repr({"credential": credential09})]
for view in views09:
    print(view)

Credential for billing
ApiCredential09(service='billing', token=<redacted>)
[ApiCredential09(service='billing', token=<redacted>)]
{'credential': ApiCredential09(service='billing', token=<redacted>)}


### Verification 09 — executable tests

In [19]:
assert all("DEMO_TOKEN_DO_NOT_LOG_42" not in view for view in views09)
assert "<redacted>" in repr(credential09)
assert "billing" in str(credential09)
assert credential09._token == "DEMO_TOKEN_DO_NOT_LOG_42"  # accessible: not a security barrier
print("PASS 09 — no token in tested display paths")

PASS 09 — no token in tested display paths


**Extension / discussion.** For full defense in depth, audit exception messages, structured logs, dataclass fields, tracing, and all other channels that might render attributes.

## Problem 10 — Exceptions: messages vs diagnostics

**Task.** Implement an exception with structured error code and path, user-facing `str`, and diagnostic `repr`. Catch and inspect a raised instance without showing a traceback.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 10 — reasoning

Exception classes can provide readable error messages while retaining concise debugging context. Explicit `__str__` makes the human-facing message predictable. The exception's `.args` behavior is separate from these representations.

In [20]:
class ConfigError10(Exception):
    def __init__(self, code: int, path: str, detail: str) -> None:
        self.code, self.path, self.detail = code, path, detail
        super().__init__(detail)

    def __repr__(self) -> str:
        return f"ConfigError10(code={self.code!r}, path={self.path!r}, detail={self.detail!r})"

    def __str__(self) -> str:
        return f"Configuration error {self.code} at {self.path}: {self.detail}"

try:
    raise ConfigError10(404, "settings.json", "missing field 'port'")
except ConfigError10 as error10:
    captured10 = error10
    print("message:", str(error10))
    print("diagnostic:", repr(error10))

message: Configuration error 404 at settings.json: missing field 'port'
diagnostic: ConfigError10(code=404, path='settings.json', detail="missing field 'port'")


### Verification 10 — executable tests

In [21]:
assert str(captured10).startswith("Configuration error 404")
assert "path='settings.json'" in repr(captured10)
assert "missing field" in str(captured10)
assert captured10.args == ("missing field 'port'",)
print("PASS 10 — exception representations")

PASS 10 — exception representations


## Problem 11 — Format-specification protocol

**Task.** First show the error produced by a nonempty format spec when `__format__` is inherited. Then implement `__format__` so that a label supports standard string alignment such as `>16` or `^20`. Test empty format, custom alignment, and `!r` conversion.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 11 — reasoning

`format(value, spec)` looks for `__format__`. A user-defined `__str__` alone does not enable arbitrary nonempty specs: `object.__format__` rejects them. Delegating `format(str(self), spec)` is a good choice when the desired spec language is the normal **string** formatting mini-language.

In [22]:
class NaiveLabel11:
    def __str__(self) -> str:
        return "ready"

class AlignedLabel11:
    def __init__(self, text: str) -> None:
        self.text = text

    def __repr__(self) -> str:
        return f"AlignedLabel11(text={self.text!r})"

    def __str__(self) -> str:
        return self.text

    def __format__(self, format_spec: str) -> str:
        return format(str(self), format_spec)

bad_format11 = captured_type_error(lambda: format(NaiveLabel11(), ">10"))
aligned11 = AlignedLabel11("ready")
print("inherited format error:", bad_format11)
print("aligned:", f"|{aligned11:>16}|")
print("centered:", f"|{aligned11:^20}|")

inherited format error: unsupported format string passed to NaiveLabel11.__format__
aligned: |           ready|
centered: |       ready        |


### Verification 11 — executable tests

In [23]:
assert "unsupported format string" in bad_format11
assert f"{aligned11}" == "ready"
assert f"{aligned11:>16}" == " " * 11 + "ready"
assert f"{aligned11:^20}" == " " * 7 + "ready" + " " * 8
assert f"{aligned11!r}" == repr(aligned11)
assert f"{aligned11!r:>30}" == format(repr(aligned11), ">30")
print("PASS 11 — __format__ and conversion precedence")

PASS 11 — __format__ and conversion precedence


**Extension / discussion.** `f"{aligned11!r:>30}"` converts the object with `repr` first, then applies the specification to the resulting *string*.

## Problem 12 — Financial output without float artifacts

**Task.** Model a decimal monetary amount with exact `Decimal` values. Give it a developer-facing repr and a friendly str. Implement `__format__` supporting `""` for the friendly display and `".3f"` for the amount alone, with the currency appended. Reject nonfinite values and reject invalid formatting specs naturally.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 12 — reasoning

For money-like data, construct `Decimal` from text instead of a binary float. Keep diagnostic output faithful to the stored value; make display rounding explicit. The custom format-spec contract below is deliberately small: a nonempty spec applies to the `Decimal` and does **not** perform whole-label alignment.

In [24]:
class Money12:
    def __init__(self, amount: str | Decimal, currency: str = "USD") -> None:
        if isinstance(amount, float):
            raise TypeError("Pass decimal text or Decimal, not float")
        try:
            parsed = Decimal(amount)
        except (InvalidOperation, TypeError, ValueError) as exc:
            raise ValueError("Invalid monetary amount") from exc
        if not parsed.is_finite():
            raise ValueError("Amount must be finite")
        if not isinstance(currency, str) or not re.fullmatch(r"[A-Z]{3}", currency):
            raise ValueError("Currency must be a three-letter uppercase code")
        self.amount, self.currency = parsed, currency

    def __repr__(self) -> str:
        return f"Money12(amount={str(self.amount)!r}, currency={self.currency!r})"

    def __str__(self) -> str:
        return f"{self.amount:.2f} {self.currency}"

    def __format__(self, format_spec: str) -> str:
        return str(self) if not format_spec else f"{format(self.amount, format_spec)} {self.currency}"

money12 = Money12("0.30")
print("developer:", repr(money12))
print("human:", money12)
print("precision:", f"{money12:.3f}")

developer: Money12(amount='0.30', currency='USD')
human: 0.30 USD
precision: 0.300 USD


### Verification 12 — executable tests

In [25]:
assert str(money12) == "0.30 USD"
assert repr(money12) == "Money12(amount='0.30', currency='USD')"
assert f"{money12:.3f}" == "0.300 USD"
assert format(money12, "") == str(money12)
for invalid in ["NaN", "Infinity", "-Infinity"]:
    try:
        Money12(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Expected rejection of {invalid}")
try:
    Money12(0.1)
except TypeError:
    pass
else:
    raise AssertionError("Binary float should be rejected")
print("PASS 12 — decimal display and input validation")

PASS 12 — decimal display and input validation


**Extension / discussion.** Production money systems additionally need an explicit rounding policy and currency-dependent minor-unit rules. This exercise focuses on representation, not accounting.

## Problem 13 — Self-referential data structures

**Task.** Demonstrate a self-referential list's built-in repr, then implement a node whose repr remains finite when its `next` pointer refers to itself. Contrast an unsafe recursive implementation (without triggering it) with a guarded one.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 13 — reasoning

Naively using `repr(self.next)` inside `__repr__` loops forever if `self.next is self`. The standard-library `reprlib.recursive_repr` decorator guards recursive invocations of **that method on the same object in the same thread** and substitutes a marker. It does not solve every possible graph-formatting concern.

In [26]:
cyclic_list13 = []
cyclic_list13.append(cyclic_list13)
print("built-in list:", repr(cyclic_list13))

class Node13:
    def __init__(self, label: str, next_node=None) -> None:
        self.label, self.next = label, next_node

    @reprlib.recursive_repr(fillvalue="<cycle>")
    def __repr__(self) -> str:
        return f"Node13(label={self.label!r}, next={self.next!r})"

    def __str__(self) -> str:
        return self.label

node13 = Node13("root")
node13.next = node13
print("safe node:", repr(node13))

built-in list: [[...]]
safe node: Node13(label='root', next=<cycle>)


### Verification 13 — executable tests

In [27]:
assert repr(cyclic_list13) == "[[...]]"
assert repr(node13) == "Node13(label='root', next=<cycle>)"
assert str(node13) == "root"
assert repr(node13) == repr(node13)
print("PASS 13 — self-reference bounded by recursion guard")

PASS 13 — self-reference bounded by recursion guard


## Problem 14 — Bounded diagnostic output with `reprlib`

**Task.** Show why enormous nested structures make poor diagnostic logs. Configure `reprlib.Repr` to shorten lists and strings, then compare bounded output with a normal representation. Leave the original object untouched.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 14 — reasoning

`reprlib` supplies shortened *presentations* that retain shape without dumping full datasets. Its output may not be valid Python and should never be parsed or used as a serialization. This is complementary to, not a replacement for, designing your object's own `__repr__`.

In [28]:
payload14 = {"values": list(range(100)), "comment": "x" * 300}
shortener14 = reprlib.Repr()
shortener14.maxlist = 5
shortener14.maxstring = 35
shortener14.maxdict = 3
normal14 = repr(payload14)
short14 = shortener14.repr(payload14)
print("normal length:", len(normal14))
print("bounded:", short14)

normal length: 717
bounded: {'comment': 'xxxxxxxxxxxxxxx...xxxxxxxxxxxxxxx', 'values': [0, 1, 2, 3, 4, ...]}


### Verification 14 — executable tests

In [29]:
assert len(short14) < len(normal14)
assert "..." in short14
assert payload14["values"] == list(range(100))
assert payload14["comment"] == "x" * 300
print("PASS 14 — bounded repr without mutation")

PASS 14 — bounded repr without mutation


## Problem 15 — Logging: `%s` versus `%r` and lazy formatting

**Task.** Log the same domain object via `%s` and `%r` using an isolated logger backed by `StringIO`. Use parameterized logging (`logger.info("... %r", obj)`) rather than preformatting an f-string.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 15 — reasoning

The logging library defers `%` interpolation until a record is actually formatted, reducing unnecessary representation work when a level is disabled. `%r` requests debugging detail, `%s` a user-oriented view. Whether a logger emits at all is controlled by its effective level and handlers.

In [30]:
class Job15:
    def __init__(self, job_id: int, state: str) -> None:
        self.job_id, self.state = job_id, state

    def __repr__(self) -> str:
        return f"Job15(job_id={self.job_id!r}, state={self.state!r})"

    def __str__(self) -> str:
        return f"job #{self.job_id}: {self.state}"

buffer15 = io.StringIO()
handler15 = logging.StreamHandler(buffer15)
handler15.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
logger15 = logging.getLogger("repr_str_exercises.job15")
logger15.handlers.clear()
logger15.addHandler(handler15)
logger15.setLevel(logging.INFO)
logger15.propagate = False
job15 = Job15(17, "queued")
logger15.info("user view: %s", job15)
logger15.info("debug view: %r", job15)
log_text15 = buffer15.getvalue()
print(log_text15, end="")

INFO user view: job #17: queued
INFO debug view: Job15(job_id=17, state='queued')


### Verification 15 — executable tests

In [31]:
assert "user view: job #17: queued" in log_text15
assert "debug view: Job15(job_id=17, state='queued')" in log_text15
assert log_text15.count("INFO") == 2
logger15.removeHandler(handler15)
handler15.close()
print("PASS 15 — controlled logging representations")

PASS 15 — controlled logging representations


## Problem 16 — Representation purity, mutation, and identity

**Task.** Show why a repr that increments a counter on every call is unsuitable for stable debugging. Implement a second class with a pure repr; change only its domain state and verify that the representation changes accordingly.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 16 — reasoning

A representation should report state, not secretly modify it or trigger expensive I/O. Calling `repr` while debugging, inspecting nested containers, or rendering in a notebook can happen more often than anticipated. Purity is a design guideline, not enforced by Python.

In [32]:
class Impure16:
    def __init__(self) -> None:
        self.calls = 0

    def __repr__(self) -> str:
        self.calls += 1
        return f"Impure16(calls={self.calls})"

class PureCounter16:
    def __init__(self, count: int = 0) -> None:
        self.count = count

    def __repr__(self) -> str:
        return f"PureCounter16(count={self.count!r})"

bad16 = Impure16()
first_bad16, second_bad16 = repr(bad16), repr(bad16)
good16 = PureCounter16(2)
first_good16, second_good16 = repr(good16), repr(good16)
good16.count += 1
updated_good16 = repr(good16)
print("impure:", first_bad16, second_bad16)
print("pure:", first_good16, second_good16, updated_good16)

impure: Impure16(calls=1) Impure16(calls=2)
pure: PureCounter16(count=2) PureCounter16(count=2) PureCounter16(count=3)


### Verification 16 — executable tests

In [33]:
assert first_bad16 != second_bad16 and bad16.calls == 2
assert first_good16 == second_good16 == "PureCounter16(count=2)"
assert updated_good16 == "PureCounter16(count=3)"
print("PASS 16 — repr side effects identified")

PASS 16 — repr side effects identified


## Problem 17 — Safe parsing of a constructor-like repr

**Task.** Implement a small, explicitly restricted parser for `Point17(x=..., y=...)` diagnostics using `ast.parse` and `ast.literal_eval` for keyword values. Refuse arbitrary calls, positional arguments, extra keyword arguments, nonnumeric values, `**kwargs`, and unexpected syntax. **Never use `eval`.**

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 17 — reasoning

An expression-like `repr` is not automatically a safe exchange format. A strict AST parser can accept one exact constructor shape without executing code. `ast.literal_eval` evaluates only literal structures, but this parser still must validate the outer call and the resulting types. Parsing untrusted input at scale also needs input-length/resource limits.

In [34]:
@dataclass(frozen=True)
class Point17:
    x: int
    y: int

    def __post_init__(self) -> None:
        if type(self.x) is not int or type(self.y) is not int:
            raise TypeError("Coordinates must be integers")


def parse_point17(source: str) -> Point17:
    if len(source) > 200:
        raise ValueError("Representation is too long")
    try:
        expression = ast.parse(source, mode="eval").body
        if not isinstance(expression, ast.Call):
            raise ValueError("Expected a constructor call")
        if not isinstance(expression.func, ast.Name) or expression.func.id != "Point17":
            raise ValueError("Only Point17 is permitted")
        if expression.args or len(expression.keywords) != 2:
            raise ValueError("Exactly two keyword arguments required")
        names = [keyword.arg for keyword in expression.keywords]
        if set(names) != {"x", "y"} or len(set(names)) != 2:
            raise ValueError("Only x and y are permitted")
        values = {keyword.arg: ast.literal_eval(keyword.value) for keyword in expression.keywords}
        if any(type(number) is not int for number in values.values()):
            raise ValueError("Coordinates must be literal integers")
        return Point17(**values)
    except (SyntaxError, TypeError, MemoryError, RecursionError) as exc:
        raise ValueError("Invalid representation") from exc

point17 = Point17(3, -8)
roundtrip17 = parse_point17(repr(point17))
print("repr:", repr(point17), "round trip:", roundtrip17)

repr: Point17(x=3, y=-8) round trip: Point17(x=3, y=-8)


### Verification 17 — executable tests

In [35]:
assert roundtrip17 == point17
malicious_or_invalid17 = [
    "__import__('os').system('echo DO_NOT_EXECUTE')",
    "Point17(x=1, y=2, z=3)",
    "Point17(1, 2)",
    "Point17(x=True, y=2)",
    "Point17(x=1, y=2, **{})",
    "Point17(x=1, x=2)",
    "Point17(x=1, y=1 / 0)",
    "Point17(x=1, y=2)" + " " * 201,
]
for source in malicious_or_invalid17:
    try:
        parse_point17(source)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Accepted invalid input: {source!r}")
print(f"PASS 17 — one valid round trip; rejected {len(malicious_or_invalid17)} invalid inputs")

PASS 17 — one valid round trip; rejected 8 invalid inputs


**Extension / discussion.** For real interchange, prefer JSON with an explicit schema over reverse-parsing a `repr`. Even safe AST parsing should not be exposed to arbitrarily large or deeply nested untrusted inputs without resource limits.

## Problem 18 — Serialization is a separate responsibility

**Task.** Demonstrate that a class's beautiful `__repr__` does not automatically make it JSON-serializable. Implement `to_dict` and `from_dict` with schema checks and perform an actual JSON round trip.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 18 — reasoning

`repr` serves Python diagnostics; JSON is a distinct data interchange standard. Give serialization a deliberate, versionable schema, rather than expecting libraries to infer it from a representation string. Only the object's explicit data dictionary is encoded.

In [36]:
@dataclass(frozen=True)
class Sensor18:
    sensor_id: str
    reading: int

    def to_dict(self) -> dict[str, object]:
        return {"sensor_id": self.sensor_id, "reading": self.reading}

    @classmethod
    def from_dict(cls, payload: dict[str, object]) -> "Sensor18":
        if not isinstance(payload, dict) or set(payload) != {"sensor_id", "reading"}:
            raise ValueError("Invalid sensor schema")
        if not isinstance(payload["sensor_id"], str) or type(payload["reading"]) is not int:
            raise ValueError("Invalid sensor types")
        return cls(payload["sensor_id"], payload["reading"])

sensor18 = Sensor18("north", 24)
serialization_error18 = captured_type_error(lambda: json.dumps(sensor18))
serialized18 = json.dumps(sensor18.to_dict(), sort_keys=True)
restored18 = Sensor18.from_dict(json.loads(serialized18))
print("repr:", repr(sensor18))
print("JSON:", serialized18)

repr: Sensor18(sensor_id='north', reading=24)
JSON: {"reading": 24, "sensor_id": "north"}


### Verification 18 — executable tests

In [37]:
assert "not JSON serializable" in serialization_error18
assert restored18 == sensor18
assert json.loads(serialized18) == {"reading": 24, "sensor_id": "north"}
for invalid in [{"sensor_id": "x"}, {"sensor_id": "x", "reading": True}, {"sensor_id": 1, "reading": 2}]:
    try:
        Sensor18.from_dict(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Bad schema accepted: {invalid!r}")
print("PASS 18 — explicit validated JSON round trip")

PASS 18 — explicit validated JSON round trip


## Problem 19 — Exhaustive edge-case testing with `itertools`

**Task.** Generate a small Cartesian product of tricky names and ages. Verify representation determinism and literal escaping across all generated instances. Add a regression check for a quote-related bug from the supplied lesson.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 19 — reasoning

Parameter matrices catch edge cases that one visually pleasant demo misses. Use assertions about meaningful properties (type, escaping, contained field representations), rather than hard-coding implementation-specific address values or relying on manual inspection.

In [38]:
edge_names19 = ["", "Alice", "O'Reilly", 'a "quote"', "line\nbreak", "\\slash", "☕"]
edge_ages19 = [0, 1, 99]
samples19 = [Person03(name, age) for name, age in itertools.product(edge_names19, edge_ages19)]
failures19 = []
for sample in samples19:
    text = repr(sample)
    if not (isinstance(text, str) and repr(sample.name) in text and f"age={sample.age}" in text):
        failures19.append(sample)
print("Cases tested:", len(samples19))
print("Example containing a quote:", repr(Person03("O'Reilly", 30)))

Cases tested: 21
Example containing a quote: Person03(name="O'Reilly", age=30)


### Verification 19 — executable tests

In [39]:
assert len(samples19) == len(edge_names19) * len(edge_ages19) == 21
assert not failures19, f"Unexpected failures: {failures19!r}"
assert repr(Person03("O'Reilly", 30)) == 'Person03(name="O\'Reilly", age=30)'
assert all(repr(p) == repr(p) for p in samples19)
print("PASS 19 — 21 combinations plus quoting regression")

PASS 19 — 21 combinations plus quoting regression


## Problem 20 — Capstone: safe, bounded, formatted audit record

**Task.** Implement an `AuditRecord` that combines (a) informative, stable diagnostic repr, (b) a readable str, (c) secret redaction, (d) bounded payload representation, (e) alignment using `__format__`, and (f) reusable `to_dict` JSON export without exposing the token. Verify these requirements through nested containers, formatting, and round-trip serialization.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 20 — reasoning

Keep output contracts separate: a *diagnostic* representation includes a bounded snapshot and redaction; a *human* view contains a short status; formatting delegates string alignment; and serialization uses a strict, explicit schema. For predictability, store an immutable tuple of integers, validate inputs before storing them, and use a controlled `reprlib` formatter.

In [40]:
class AuditRecord20:
    def __init__(self, record_id: str, status: str, values: list[int], token: str) -> None:
        if not isinstance(record_id, str) or not record_id:
            raise ValueError("record_id must be nonempty text")
        if status not in {"new", "approved", "rejected"}:
            raise ValueError("unsupported status")
        if not isinstance(values, list) or any(type(v) is not int for v in values):
            raise TypeError("values must be a list of integers")
        if not isinstance(token, str):
            raise TypeError("token must be text")
        self.record_id = record_id
        self.status = status
        self._values = tuple(values)  # caller cannot mutate this via its original list
        self._token = token

    def __repr__(self) -> str:
        formatter = reprlib.Repr()
        formatter.maxtuple = 4
        return (
            f"AuditRecord20(record_id={self.record_id!r}, status={self.status!r}, "
            f"values={formatter.repr(self._values)}, token=<redacted>)"
        )

    def __str__(self) -> str:
        return f"Audit {self.record_id}: {self.status} ({len(self._values)} values)"

    def __format__(self, format_spec: str) -> str:
        return format(str(self), format_spec)

    def to_dict(self) -> dict[str, object]:
        return {"schema_version": 1, "record_id": self.record_id,
                "status": self.status, "values": list(self._values)}

    @classmethod
    def from_dict(cls, payload: dict[str, object], *, token: str = "") -> "AuditRecord20":
        if not isinstance(payload, dict) or set(payload) != {"schema_version", "record_id", "status", "values"}:
            raise ValueError("unexpected audit schema")
        if type(payload["schema_version"]) is not int or payload["schema_version"] != 1:
            raise ValueError("unsupported schema version")
        return cls(payload["record_id"], payload["status"], payload["values"], token)

caller_values20 = list(range(30))
audit20 = AuditRecord20("A-007", "approved", caller_values20, "DEMO_SUPER_SECRET_20")
caller_values20.append(999)
print("human:", audit20)
print("diagnostic:", repr(audit20))
print("aligned:", f"|{audit20:>50}|")
print("nested:", {"last": [audit20]})
json_text20 = json.dumps(audit20.to_dict(), sort_keys=True)
print("JSON preview:", json_text20[:100] + "...")

human: Audit A-007: approved (30 values)
diagnostic: AuditRecord20(record_id='A-007', status='approved', values=(0, 1, 2, 3, ...), token=<redacted>)
aligned: |                 Audit A-007: approved (30 values)|
nested: {'last': [AuditRecord20(record_id='A-007', status='approved', values=(0, 1, 2, 3, ...), token=<redacted>)]}
JSON preview: {"record_id": "A-007", "schema_version": 1, "status": "approved", "values": [0, 1, 2, 3, 4, 5, 6, 7,...


### Verification 20 — executable tests

In [41]:
views20 = [str(audit20), repr(audit20), f"{audit20:>50}", repr({"last": [audit20]}), json_text20]
assert all("DEMO_SUPER_SECRET_20" not in text for text in views20)
assert "<redacted>" in repr(audit20)
assert "..." in repr(audit20)
assert "999" not in repr(audit20) and len(audit20._values) == 30
assert len(f"{audit20:>50}") >= 50
assert json.loads(json_text20)["schema_version"] == 1
assert json.loads(json_text20)["values"] == list(range(30))
restored20 = AuditRecord20.from_dict(json.loads(json_text20), token="new-secret")
assert restored20.to_dict() == audit20.to_dict()
assert "new-secret" not in repr(restored20)
try:
    AuditRecord20.from_dict({"schema_version": 99, "record_id": "x", "status": "new", "values": []})
except ValueError:
    pass
else:
    raise AssertionError("Unknown schema version accepted")
print("PASS 20 — capstone representations, formatting, redaction, JSON")

PASS 20 — capstone representations, formatting, redaction, JSON


**Extension / discussion.** The `token=<redacted>` fragment intentionally makes the repr *non-executable*. For sensitive objects, diagnostic clarity and avoiding secrets take precedence over reconstructible expression syntax.

---
# Eight further advanced problems — complete worked solutions

Attempt each task first, then compare with the reference solution and run its assertions. No exercise is left without a solution.

## Problem 21 — Inherited repr and overridden str

**Task.** Derive a child that inherits its parent's `__repr__` but overrides `__str__`. Compare direct display, `!r`, and display inside a list.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 21 — reasoning

An inherited repr remains active unless a subclass overrides it. Containers display element reprs, not element strs. A parent-named repr is only a good choice when inherited state is still sufficient.

In [42]:
class Parent21:
    def __init__(self, value: str) -> None:
        self.value = value

    def __repr__(self) -> str:
        return f"Parent21(value={self.value!r})"

class Child21(Parent21):
    def __str__(self) -> str:
        return f"friendly: {self.value}"

child21 = Child21("alpha")
print("repr:", repr(child21))
print("str:", str(child21))
print("!r:", f"{child21!r}")
print("list:", [child21])

repr: Parent21(value='alpha')
str: friendly: alpha
!r: Parent21(value='alpha')
list: [Parent21(value='alpha')]


### Verification 21 — executable tests

In [43]:
assert repr(child21) == "Parent21(value='alpha')"
assert str(child21) == "friendly: alpha"
assert f"{child21!r}" == repr(child21)
assert repr([child21]) == "[Parent21(value='alpha')]"
assert Child21.__repr__ is Parent21.__repr__
print("PASS 21 — inheritance and conversion context")

PASS 21 — inheritance and conversion context


## Problem 22 — A strict formatting mini-language

**Task.** Accept only empty, `>N`, and `^N` string-format specs with width 0–80. Reject other specs using a descriptive `ValueError`.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 22 — reasoning

Whitelist the accepted grammar, check the width, and then delegate to Python's standard string formatting. This avoids accidentally promising support for unwanted formatting flags.

In [44]:
class RestrictedLabel22:
    def __init__(self, label: str) -> None:
        self.label = label

    def __repr__(self) -> str:
        return f"RestrictedLabel22(label={self.label!r})"

    def __str__(self) -> str:
        return self.label

    def __format__(self, spec: str) -> str:
        if not spec:
            return str(self)
        if not re.fullmatch(r"[>^](?:0|[1-9][0-9]?)", spec):
            raise ValueError("Use >N or ^N with N in 0..80")
        if int(spec[1:]) > 80:
            raise ValueError("Width exceeds 80")
        return format(str(self), spec)

label22 = RestrictedLabel22("go")
print(repr(format(label22, ">6")), repr(format(label22, "^9")))

'    go' '   go    '


### Verification 22 — executable tests

In [45]:
assert format(label22, "") == "go"
assert format(label22, ">6") == "    go"
assert format(label22, "^9") == "   go    "
for spec in ["<8", "-2", ">81", ">999", ">4.2", "x", "*>8"]:
    try:
        format(label22, spec)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Unexpectedly accepted {spec!r}")
print("PASS 22 — restricted format grammar")

PASS 22 — restricted format grammar


## Problem 23 — Three-level inheritance and complete repr

**Task.** Extend Problem 7 with a timezone field. Include all constructor arguments in the new subclass repr, and show an explicit reconstruction without executing the repr as code.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 23 — reasoning

An inherited repr may omit state added in a subclass. Implement a new repr listing each field with `!r`, and reconstruct via explicit attribute access. Do not use `eval`.

In [46]:
class ZonedEvent23(TimedEvent07):
    def __init__(self, title: str, timestamp: str, timezone: str) -> None:
        super().__init__(title, timestamp)
        if not timezone:
            raise ValueError("Timezone is required")
        self.timezone = timezone

    def __repr__(self) -> str:
        return (f"ZonedEvent23(title={self.title!r}, "
                f"timestamp={self.timestamp!r}, timezone={self.timezone!r})")

    def __str__(self) -> str:
        return f"{super().__str__()} [{self.timezone}]"

zoned23 = ZonedEvent23("release", "2026-09-19T10:15:00", "Europe/Sofia")
recreated23 = ZonedEvent23(zoned23.title, zoned23.timestamp, zoned23.timezone)
print(repr(zoned23))
print(str(zoned23))

ZonedEvent23(title='release', timestamp='2026-09-19T10:15:00', timezone='Europe/Sofia')
release @ 2026-09-19T10:15:00 [Europe/Sofia]


### Verification 23 — executable tests

In [47]:
assert isinstance(zoned23, TimedEvent07)
assert repr(recreated23) == repr(zoned23)
assert "timestamp='2026-09-19T10:15:00'" in repr(zoned23)
assert "timezone='Europe/Sofia'" in repr(zoned23)
assert "[Europe/Sofia]" in str(zoned23)
print("PASS 23 — complete subclass representation")

PASS 23 — complete subclass representation


## Problem 24 — Find secret leakage through nested metadata

**Task.** Show a leak caused by a dataclass with `token=field(repr=False)` but a repr-visible metadata dict containing the same token. Then define a safe allowlisted repr, str, and public JSON dictionary.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 24 — reasoning

Marking one field `repr=False` is insufficient if a separate repr-visible field contains sensitive data. Explicit allowlists reduce accidental exposure but are not an access-control or encryption mechanism.

In [48]:
@dataclass
class Leaky24:
    service: str
    token: str = field(repr=False)
    metadata: dict[str, str] = field(default_factory=dict)

leaky24 = Leaky24("payments", "DEMO_SECRET_24", {"backup": "DEMO_SECRET_24"})
leak_detected24 = leaky24.token in repr(leaky24)

@dataclass(repr=False)
class Safe24:
    service: str
    token: str = field(repr=False)
    metadata: dict[str, str] = field(default_factory=dict, repr=False)

    def __repr__(self) -> str:
        return f"Safe24(service={self.service!r}, token=<redacted>, metadata=<omitted>)"

    def __str__(self) -> str:
        return f"Credential for {self.service}"

    def public_dict(self) -> dict[str, str]:
        return {"service": self.service}

safe24 = Safe24(leaky24.service, leaky24.token, leaky24.metadata)
outputs24 = [str(safe24), repr(safe24), repr([safe24]), json.dumps(safe24.public_dict())]
print("Detected nested leak:", leak_detected24)
print("Safe diagnostic:", repr(safe24))

Detected nested leak: True
Safe diagnostic: Safe24(service='payments', token=<redacted>, metadata=<omitted>)


### Verification 24 — executable tests

In [49]:
assert leak_detected24
assert all("DEMO_SECRET_24" not in output for output in outputs24)
assert safe24.public_dict() == {"service": "payments"}
assert safe24.token == "DEMO_SECRET_24"  # hiding display is not access control
print("PASS 24 — nested redaction regression")

PASS 24 — nested redaction regression


## Problem 25 — Mutually recursive object graphs

**Task.** Make node A reference node B and B reference A using Problem 13. Predict both repr strings exactly and check graph identity remains unchanged.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 25 — reasoning

The `recursive_repr` decorator substitutes the cycle marker upon re-entering repr for a node already active on the same thread. The graph is not modified.

In [50]:
a25 = Node13("a")
b25 = Node13("b")
a25.next = b25
b25.next = a25
print("A:", repr(a25))
print("B:", repr(b25))

A: Node13(label='a', next=Node13(label='b', next=<cycle>))
B: Node13(label='b', next=Node13(label='a', next=<cycle>))


### Verification 25 — executable tests

In [51]:
assert repr(a25) == "Node13(label='a', next=Node13(label='b', next=<cycle>))"
assert repr(b25) == "Node13(label='b', next=Node13(label='a', next=<cycle>))"
assert a25.next is b25 and b25.next is a25
print("PASS 25 — two-node cycle")

PASS 25 — two-node cycle


## Problem 26 — Bounded output for ten thousand values

**Task.** Compare the number of characters produced by ordinary repr and `reprlib` on 10,000 integers. Avoid timing assertions: environment-dependent speed is not the point of this test.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 26 — reasoning

Representation size is deterministic for fixed data. Bounded output preserves a preview without dumping huge data structures into logging or notebook outputs.

In [52]:
numbers26 = list(range(10_000))
full26 = repr(numbers26)
formatter26 = reprlib.Repr()
formatter26.maxlist = 6
short26 = formatter26.repr(numbers26)
print({"elements": len(numbers26), "full_chars": len(full26),
       "short_chars": len(short26), "preview": short26})

{'elements': 10000, 'full_chars': 58890, 'short_chars': 23, 'preview': '[0, 1, 2, 3, 4, 5, ...]'}


### Verification 26 — executable tests

In [53]:
assert len(full26) > 50_000
assert len(short26) < 100
assert "..." in short26
assert numbers26[-1] == 9999
print("PASS 26 — bounded diagnostic volume")

PASS 26 — bounded diagnostic volume


## Problem 27 — Numeric width versus whole-label width

**Task.** Why does `f"{money12:>20}"` differ from `f"{money12!s:>20}"`? Create a subclass supporting a `label:` prefix for whole-label alignment without affecting numeric specs such as `.3f`.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 27 — reasoning

The existing `Money12.__format__` formats the Decimal amount and appends the currency; `!s` converts the whole object to a string *before* string alignment. An explicit new prefix avoids ambiguous interpretation.

In [54]:
class AlignedMoney27(Money12):
    def __repr__(self) -> str:
        return f"AlignedMoney27(amount={str(self.amount)!r}, currency={self.currency!r})"

    def __format__(self, spec: str) -> str:
        if spec.startswith("label:"):
            return format(str(self), spec[len("label:"):])
        return super().__format__(spec)

money27 = AlignedMoney27("12.30", "EUR")
numeric27 = f"{money27:>20}"
whole27 = f"{money27:label:>20}"
converted27 = f"{money27!s:>20}"
print("numeric:", repr(numeric27))
print("whole:", repr(whole27))

numeric: '               12.30 EUR'
whole: '           12.30 EUR'


### Verification 27 — executable tests

In [55]:
assert numeric27 == " " * 15 + "12.30 EUR"
assert whole27 == converted27 == " " * 11 + "12.30 EUR"
assert f"{money27:.3f}" == "12.300 EUR"
assert repr(money27) == "AlignedMoney27(amount='12.30', currency='EUR')"
print("PASS 27 — explicit whole-label alignment")

PASS 27 — explicit whole-label alignment


## Problem 28 — Versioned schema migration

**Task.** Introduce audit schema v2 with a required `actor`. Accept v1 using a documented default actor, accept v2 with strict keys, reject unknown versions, and omit all tokens from public JSON.

**Think first.** Write down the predicted behavior before running the solution.

**Success criteria.** The solution runs top-to-bottom with no third-party packages and all assertions pass.

### Solution 28 — reasoning

Serialization and repr serve different purposes. Define a strict v2 schema and reuse v1 validation during migration. Supply private tokens out-of-band; never recover them from public JSON.

In [56]:
class AuditV2_28:
    def __init__(self, record: AuditRecord20, actor: str) -> None:
        if not isinstance(record, AuditRecord20) or not isinstance(actor, str) or not actor:
            raise ValueError("A record and nonempty actor are required")
        self.record, self.actor = record, actor

    def __repr__(self) -> str:
        return f"AuditV2_28(record={self.record!r}, actor={self.actor!r})"

    def __str__(self) -> str:
        return f"{self.record} by {self.actor}"

    def to_dict(self) -> dict[str, object]:
        result = self.record.to_dict()
        result["schema_version"] = 2
        result["actor"] = self.actor
        return result

    @classmethod
    def from_dict(cls, payload: dict[str, object], *, token: str = "") -> "AuditV2_28":
        if not isinstance(payload, dict) or type(payload.get("schema_version")) is not int:
            raise ValueError("Missing or invalid schema version")
        version = payload["schema_version"]
        if version == 1:
            return cls(AuditRecord20.from_dict(payload, token=token), "unknown")
        if version != 2:
            raise ValueError("Unsupported schema version")
        if set(payload) != {"schema_version", "record_id", "status", "values", "actor"}:
            raise ValueError("Unexpected v2 keys")
        if not isinstance(payload["actor"], str) or not payload["actor"]:
            raise ValueError("Invalid actor")
        old = {key: value for key, value in payload.items() if key != "actor"}
        old["schema_version"] = 1
        return cls(AuditRecord20.from_dict(old, token=token), payload["actor"])

migrated28 = AuditV2_28.from_dict(audit20.to_dict(), token="private-migrated")
v2_json28 = json.dumps(AuditV2_28(audit20, "operator-1").to_dict())
restored28 = AuditV2_28.from_dict(json.loads(v2_json28), token="private-v2")
print("migrated actor:", migrated28.actor)
print("v2 repr:", repr(restored28))

migrated actor: unknown
v2 repr: AuditV2_28(record=AuditRecord20(record_id='A-007', status='approved', values=(0, 1, 2, 3, ...), token=<redacted>), actor='operator-1')


### Verification 28 — executable tests

In [57]:
assert migrated28.actor == "unknown"
assert migrated28.to_dict()["schema_version"] == 2
assert restored28.actor == "operator-1"
assert json.loads(v2_json28) == restored28.to_dict()
assert "private-migrated" not in repr(migrated28)
assert "private-v2" not in repr(restored28)
for invalid in [dict(restored28.to_dict(), schema_version=3),
                dict(restored28.to_dict(), actor=""),
                dict(restored28.to_dict(), unexpected=1)]:
    try:
        AuditV2_28.from_dict(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Accepted invalid schema: {invalid!r}")
print("PASS 28 — v1/v2 migration and strict validation")

PASS 28 — v1/v2 migration and strict validation


## Final self-check and key takeaways

- `__repr__` aims for precise debugging; `__str__` aims for readable display.
- If `__str__` is absent, the inherited default delegates to `__repr__`; the reverse does not happen.
- The Python representation methods **must return strings**. Use `!r` for values embedded in constructor-like expressions.
- Containers normally use element reprs; `!r`, `!s`, `!a` provide explicit conversion control.
- For nonempty format specs, define `__format__` and document which mini-language you support.
- Representations should generally be pure, useful, bounded where necessary, and careful with secrets.
- Repr is not a serializer and not a security boundary. Prefer JSON (or another deliberate serialization format) for exchange; never blindly `eval(repr(x))`.

**Completion:** All 28 worked problems have executable verification cells; a clean *Restart & Run All* should finish without assertion failures. The eight additional challenges are intentionally unsolved for independent practice.